In [ ]:
import os
import json
import copy
import random
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler

from IPython.display import display
from IPython import get_ipython 

# Optional GPU acceleration for supported sklearn estimators
USE_GPU_ACCEL = True
GPU_ACCEL_ACTIVE = False

if USE_GPU_ACCEL:
    try:
        ip = get_ipython()

        if ip is not None:
            ip.run_line_magic("load_ext", "cuml.accel")
        else:
            import cuml.accel  # requires RAPIDS/cuML installed in this interpreter

        # Re-import sklearn estimators after enabling cuml.accel
        from sklearn.decomposition import PCA
        from sklearn.impute import SimpleImputer, KNNImputer
        from sklearn.linear_model import Ridge, ElasticNet
        from sklearn.preprocessing import StandardScaler

        GPU_ACCEL_ACTIVE = True
        print("[GPU] cuML accel enabled.")
    except Exception as e:
        GPU_ACCEL_ACTIVE = False
        print(f"[WARN] cuML accel not enabled, falling back to CPU: {e}")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# Seeds 
BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]

SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

# Paths 
ARTIFACTS = Path("artifacts")
IN_CLEAN  = ARTIFACTS / "cleaned"  / "notebook 2"
IN_T2     = ARTIFACTS / "aligned"  / "notebook 1" / "track2_nonintersection"
IN_META   = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 4"
OUT_REPORTS = ARTIFACTS / "reports"  / NOTEBOOK_SUBDIR
OUT_META    = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META]:
    d.mkdir(parents=True, exist_ok=True)

# Load backbone lock 
LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(
        f"proteomics_backbone_lock.json not found at {LOCK_PATH}. "
        "Run the lock cell at the end of Notebook 3b first."
    )
with LOCK_PATH.open("r") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM   = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM    = backbone_lock["track2_stress_test_arm"]

print("Backbone lock loaded.")
print(f"  Primary arm  : {PRIMARY_ARM}")
print(f"  Secondary arm: {SECONDARY_ARM}")
print(f"  Track 2 arm  : {TRACK2_ARM}")
print(f"  Seeds to run : {ALL_SEEDS}")

# Exact top-10 ranked configurations from Notebook 3b
TOP10_CONFIGS = [
    {"rank": 1,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna"},
    {"rank": 2,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+prot"},
    {"rank": 3,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna"},
    {"rank": 4,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+mut"},
    {"rank": 5,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 6,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+prot"},
    {"rank": 7,  "arm": "prot_procan_depmapSanger", "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 8,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+mut"},
    {"rank": 9,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 10, "arm": "prot_procan_depmapSanger", "model": "elasticnet", "feature_set": "prot"},
]

# Arms now come from the actual top-10 rows, not just the lock file
ACTIVE_ARMS = sorted({cfg["arm"] for cfg in TOP10_CONFIGS})

CONFIGS_BY_ARM = {
    arm: [cfg for cfg in TOP10_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

print("\nTop-10 ranked configs loaded.")
display(pd.DataFrame(TOP10_CONFIGS))
print("Active arms from top-10 configs:", ACTIVE_ARMS)

# Bake-off settings 
N_DRUGS_BAKEOFF    = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED   = 10
RIDGE_ALPHA        = 1.0
EN_ALPHA           = 0.05
EN_L1_RATIO        = 0.2
PCA_COMPONENTS     = 100
KNN_NEIGHBORS      = 5
PRIMARY_TARGET     = "auc"

[WARN] cuML accel not enabled, falling back to CPU: No module named 'cuml'
Backbone lock loaded.
  Primary arm  : prot_procan_depmapSanger
  Secondary arm: prot_ms_ccle_gygi
  Track 2 arm  : prot_combined_union
  Seeds to run : [19537, 1584678, 17052356]

Top-10 ranked configs loaded.


,rank,arm,model,feature_set
0,1,prot_combined_union,elasticnet,rna
1,2,prot_rppa_ccle,elasticnet,rna+prot
2,3,prot_rppa_ccle,elasticnet,rna
3,4,prot_combined_union,elasticnet,rna+mut
4,5,prot_combined_union,elasticnet,rna+mut+prot
5,6,prot_combined_union,elasticnet,rna+prot
6,7,prot_procan_depmapSanger,elasticnet,rna+mut+prot
7,8,prot_rppa_ccle,elasticnet,rna+mut
8,9,prot_rppa_ccle,elasticnet,rna+mut+prot
9,10,prot_procan_depmapSanger,elasticnet,prot


Active arms from top-10 configs: ['prot_combined_union', 'prot_procan_depmapSanger', 'prot_rppa_ccle']


In [6]:
#  Helpers 
def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)

def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])

def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"

def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int,
    seed: int,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    groups   = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups = groups.nunique()
    n_cells  = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        sp = GroupKFold(n_splits=n_splits)
        return (
            list(sp.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)),
            f"GroupKFold(n_splits={n_splits})",
        )
    n_splits = min(max(2, n_splits), n_cells)
    sp = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sp.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits})"

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    if feature_set == "":
        return tuple()
    return tuple(feature_set.split("+"))

def concat_selected_modalities(
    mats: Dict[str, np.ndarray],
    selected_keys: Tuple[str, ...],
    n_rows: int,
) -> np.ndarray:
    parts = []
    for k in selected_keys:
        if k == "prot":
            continue
        arr = mats.get(k)
        if arr is None or arr.shape[1] == 0:
            return np.zeros((n_rows, 0), dtype=np.float32)
        parts.append(arr)
    if not parts:
        return np.zeros((n_rows, 0), dtype=np.float32)
    return np.concatenate(parts, axis=1)

def make_model(model_name: str, seed: int):
    model_name = str(model_name).lower()
    if model_name == "ridge":
        return Ridge(alpha=RIDGE_ALPHA, random_state=seed)
    if model_name == "elasticnet":
        return ElasticNet(
            alpha=EN_ALPHA,
            l1_ratio=EN_L1_RATIO,
            random_state=seed,
            max_iter=10000,
        )
    raise ValueError(f"Unsupported model for Notebook 4 bake-off: {model_name}")

# Load backbone data (once — shared across all seeds) 
cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col  = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string").fillna("Unknown").astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print(f"\nCore cohort (RNA ∩ CNV ∩ MUT): {len(core_cells)} cell lines")
print(f"Group column for CV: {group_col}")

prism_long["depmap_id"]   = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"]      = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][
    ["depmap_id", "compound_id", "y"]
].copy()

# Load locked proteomics arms 
prot_arm_data: Dict[str, pd.DataFrame] = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} not found at {p}, skipping.")

# Drug selection (fixed by coverage, not performance) 
drug_cov = (
    prism_auc.groupby("compound_id")["depmap_id"]
    .nunique().sort_values(ascending=False)
)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel      = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs  = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print(f"\nSelected {len(selected_drugs)} drugs for bake-off by PRISM AUC coverage.")




Core cohort (RNA ∩ CNV ∩ MUT): 1079 cell lines
Group column for CV: lineage_1
Loaded prot_combined_union: (1079, 18751)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)

Selected 100 drugs for bake-off by PRISM AUC coverage.


In [7]:
# MISSINGNESS REPORT  (seed-independent)
print("MISSINGNESS REPORT")


# Per-modality availability
avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()

availability = pd.DataFrame(avail_rows, index=core_cells)
availability.index.name = "depmap_id"
avail_path = OUT_REPORTS / "modality_availability_matrix.csv"
availability.to_csv(avail_path)
print(f"Availability matrix: {avail_path}  shape={availability.shape}")

avail_summary = (
    availability.sum().rename("n_cells_present").to_frame()
    .assign(
        n_cells_total=len(core_cells),
        pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2),
    )
)
avail_summary_path = OUT_REPORTS / "modality_availability_summary.csv"
avail_summary.to_csv(avail_summary_path)
print("\nModality availability summary:")
display(avail_summary)

# Per-arm feature-level missingness
feat_miss_rows = []
for arm, df in prot_arm_data.items():
    df_core  = df.reindex(core_cells)
    col_miss = df_core.isna().mean()
    row_miss = df_core.isna().mean(axis=1)
    feat_miss_rows.append({
        "arm": arm,
        "n_features": int(df_core.shape[1]),
        "n_cells_in_core": int(df_core.shape[0]),
        "overall_missing_pct": float(df_core.isna().mean().mean() * 100),
        "col_miss_q10": float(col_miss.quantile(0.10)),
        "col_miss_q25": float(col_miss.quantile(0.25)),
        "col_miss_q50": float(col_miss.quantile(0.50)),
        "col_miss_q75": float(col_miss.quantile(0.75)),
        "col_miss_q90": float(col_miss.quantile(0.90)),
        "col_miss_q99": float(col_miss.quantile(0.99)),
        "row_miss_q10": float(row_miss.quantile(0.10)),
        "row_miss_q50": float(row_miss.quantile(0.50)),
        "row_miss_q90": float(row_miss.quantile(0.90)),
        "n_cols_fully_missing": int((col_miss == 1.0).sum()),
        "n_rows_fully_missing": int((row_miss == 1.0).sum()),
        "n_cols_zero_missing":  int((col_miss == 0.0).sum()),
    })
feat_miss_df   = pd.DataFrame(feat_miss_rows)
feat_miss_path = OUT_REPORTS / "per_arm_feature_missingness.csv"
feat_miss_df.to_csv(feat_miss_path, index=False)
print("\nPer-arm feature missingness:")
display(feat_miss_df)

# Combined union platform patterns
pat_counts       = None
platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df  = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes  = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0

    def pattern_label(row) -> str:
        parts = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(parts) if parts else "none"

    pat_counts = (
        plat_pres.apply(pattern_label, axis=1).rename("pattern")
        .value_counts().rename_axis("pattern").reset_index(name="n_cells")
    )
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    print("\nCombined union platform patterns:")
    display(pat_counts)

    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols:
            continue
        n_absent      = int(plat_pres[key].eq(0).sum())
        miss_from_abs = n_absent * len(cols)
        miss_total    = int(union_df[cols].isna().sum().sum())
        pm_rows.append({
            "platform": key,
            "n_features": len(cols),
            "n_cells_present": int(plat_pres[key].sum()),
            "n_cells_absent": n_absent,
            "frac_cells_present": float(plat_pres[key].mean()),
            "pct_missingness_from_platform_absence": (
                float(miss_from_abs / miss_total * 100) if miss_total > 0 else np.nan
            ),
        })
    platform_miss_df = pd.DataFrame(pm_rows)
    platform_miss_df.to_csv(
        OUT_REPORTS / "combined_union_platform_missingness_contrib.csv", index=False
    )
    print("\nCombined union, missingness from platform absence:")
    display(platform_miss_df)

def build_missingness_report(
    avail_summary: pd.DataFrame,
    feat_miss_df: pd.DataFrame,
    pat_counts: Optional[pd.DataFrame],
    platform_miss_df: Optional[pd.DataFrame],
    track2_arm: str,
    all_seeds: List[int],
    ts: str,
) -> dict:
    report = {
        "generated_at": ts,
        "seeds": all_seeds,
        "active_arms": ACTIVE_ARMS,
        "deprioritised_arm": "prot_rppa_ccle",
        "leakage_note": (
            "All imputation statistics are computed inside CV folds only. "
            "No global statistics are used at any point."
        ),
        "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
        "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
    }

    if pat_counts is not None:
        report["combined_union_platform_patterns"] = {
            "arm": track2_arm,
            "structural_missingness_warning": (
                "Missingness is driven by platform availability, not random "
                "per-protein failure. Missingness indicators are mandatory for "
                "this arm regardless of bake-off result."
            ),
            "patterns": pat_counts.to_dict(orient="records"),
        }

    if platform_miss_df is not None:
        report["combined_union_platform_missingness_contrib"] = (
            platform_miss_df.to_dict(orient="records")
        )

    report["bakeoff_outputs"] = {
        "detail_file":  f"impute_bakeoff_merged_{len(all_seeds)}seeds.csv",
        "summary_file": f"impute_bakeoff_summary_merged_{len(all_seeds)}seeds.csv",
        "lock_file":    "imputer_choice.json",
    }

    return report

missingness_report = build_missingness_report(
    avail_summary    = avail_summary,
    feat_miss_df     = feat_miss_df,
    pat_counts       = pat_counts,
    platform_miss_df = platform_miss_df,
    track2_arm       = TRACK2_ARM,
    all_seeds        = ALL_SEEDS,
    ts               = datetime.now(timezone.utc).isoformat(),
)

report_path = OUT_REPORTS / "missingness_report.json"
write_json(missingness_report, report_path)
print(f"\nMissingness report written: {report_path}")



MISSINGNESS REPORT
Availability matrix: artifacts\reports\notebook 4\modality_availability_matrix.csv  shape=(1079, 6)

Modality availability summary:


,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72



Per-arm feature missingness:


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
1,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
2,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0



Combined union platform patterns:


,pattern,n_cells,frac_cells
0,none,400,0.370714
1,ms+rppa+procan,233,0.215941
2,rppa+procan,187,0.173309
3,rppa,128,0.118628
4,ms+rppa,64,0.059314
5,procan,60,0.055607
6,ms+procan,5,0.004634
7,ms,2,0.001854



Combined union, missingness from platform absence:


,platform,n_features,n_cells_present,n_cells_absent,frac_cells_present,pct_missingness_from_platform_absence
0,ms,10847,304,775,0.281742,92.892390
1,rppa,144,612,467,0.567192,100.000000
2,procan,7760,485,594,0.449490,78.405864



Missingness report written: artifacts\reports\notebook 4\missingness_report.json


In [ ]:
# IMPUTATION BAKE-OFF (3 seeds, leakage-safe)
print("IMPUTATION BAKE-OFF (3 seeds)")

IMPUTER_STRATEGIES = [
    ("median",            SimpleImputer(strategy="median"),       False),
    ("mean",              SimpleImputer(strategy="mean"),         False),
    ("median+indicators", SimpleImputer(strategy="median"),       True),
    ("mean+indicators",   SimpleImputer(strategy="mean"),         True),
    ("knn",           KNNImputer(n_neighbors=KNN_NEIGHBORS),  False),
    ("knn+indicators",KNNImputer(n_neighbors=KNN_NEIGHBORS),  True),
]

class BaseModalityTransformer:
    def __init__(self, n_components: int, random_state: int):
        self.n_components = n_components
        self.random_state = random_state
        self._imp    = SimpleImputer(strategy="median")
        self._scaler = StandardScaler()
        self._pca    = None
        self._keep   = None

    def fit(self, X: np.ndarray) -> "BaseModalityTransformer":
        X = np.asarray(X, dtype=float)
        self._keep = np.isfinite(X).any(axis=0)
        if self._keep.sum() == 0:
            return self
        Xk = X[:, self._keep]
        Xs = self._scaler.fit_transform(self._imp.fit_transform(Xk))
        n, d   = Xs.shape
        n_comp = min(self.n_components, max(1, n - 1), d)
        self._pca = PCA(n_components=n_comp, random_state=self.random_state)
        self._pca.fit(Xs)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self._keep is None or self._keep.sum() == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        Xk = X[:, self._keep]
        Xs = self._scaler.transform(self._imp.transform(Xk))
        if self._pca is not None:
            Xs = self._pca.transform(Xs)
        return Xs.astype(np.float32)


def fit_transform_imputer(
    X_train: np.ndarray,
    X_all: np.ndarray,
    imputer,
    add_indicators: bool,
    force_indicators: bool,
    random_state: int,
    n_pca_components: int,
) -> np.ndarray:
    X_train = np.asarray(X_train, dtype=float)
    X_all   = np.asarray(X_all,   dtype=float)
    if X_train.shape[1] == 0:
        return np.zeros((X_all.shape[0], 0), dtype=np.float32)
    keep = np.isfinite(X_train).any(axis=0)
    if keep.sum() == 0:
        return np.zeros((X_all.shape[0], 0), dtype=np.float32)

    Xtr_k = X_train[:, keep]
    Xal_k = X_all[:, keep]

    use_ind = add_indicators or force_indicators
    if use_ind:
        ind_tr = np.isnan(Xtr_k).astype(np.float32)
        ind_al = np.isnan(Xal_k).astype(np.float32)
        ind_any = ind_tr.any(axis=0)
        ind_tr  = ind_tr[:, ind_any]
        ind_al  = ind_al[:, ind_any]

    imputer.fit(Xtr_k)
    scaler  = StandardScaler()
    Xtr_std = scaler.fit_transform(imputer.transform(Xtr_k))
    Xal_std = scaler.transform(imputer.transform(Xal_k))

    if use_ind and ind_tr.shape[1] > 0:
        ind_sc = StandardScaler()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            Xtr_std = np.concatenate([Xtr_std, ind_sc.fit_transform(ind_tr)], axis=1)
            Xal_std = np.concatenate([Xal_std, ind_sc.transform(ind_al)],     axis=1)

    n, d   = Xtr_std.shape
    n_comp = min(n_pca_components, max(1, n - 1), d)
    pca    = PCA(n_components=n_comp, random_state=random_state)
    pca.fit(Xtr_std)
    return pca.transform(Xal_std).astype(np.float32)


# Seed loop 
all_bakeoff_rows: List[dict] = []
seeds_to_run: List[int] = []

REQUIRED_NEW_COLS = {"config_rank", "model", "feature_set", "uses_prot"}

for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"impute_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)

        # Ignore stale outputs from the old arm-only notebook version
        if not REQUIRED_NEW_COLS.issubset(existing.columns):
            print(f"[resume] Seed {run_seed} file exists but is from old schema, rerunning.")
            seeds_to_run.append(run_seed)
            continue

        existing["seed"] = existing["seed"].astype(int)
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping bake-off loop.")
else:
    print(f"[resume] Seeds remaining: {seeds_to_run}")

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded.")
            continue

        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            print(f"  [SKIP] {arm} has no configs assigned.")
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_ckpt_path = OUT_REPORTS / f"impute_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs: set = set()

        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)

            # Ignore stale outputs from the old arm-only notebook version
            if REQUIRED_NEW_COLS.issubset(arm_existing.columns):
                arm_existing["seed"] = arm_existing["seed"].astype(int)
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    already_done_drugs = set(arm_existing["compound_id"].unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(
            eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed
        )
        print(f"  {arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig = prot_core.loc[eligible_cells]

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            if drug in already_done_drugs:
                continue

            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1

            if n_run % 5 == 0:
                arm_rows_partial = [
                    r for r in all_bakeoff_rows
                    if r["seed"] == run_seed and r["arm"] == arm
                ]
                if arm_rows_partial:
                    pd.DataFrame(arm_rows_partial).to_csv(arm_ckpt_path, index=False)
                    print(f"    [mid-checkpoint] drug {n_run}/{N_DRUGS_BAKEOFF} — {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, (train_idx, _) in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                train_cells = [eligible_cells[int(j)] for j in train_idx]

                idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)
                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                rna_tr = BaseModalityTransformer(PCA_COMPONENTS, SEED + 0).fit(
                    rna.loc[train_cells].to_numpy()
                )
                cnv_tr = BaseModalityTransformer(PCA_COMPONENTS, SEED + 1).fit(
                    cnv.loc[train_cells].to_numpy()
                )
                mut_tr = BaseModalityTransformer(PCA_COMPONENTS, SEED + 2).fit(
                    mut.loc[train_cells].to_numpy()
                )

                Xr = rna_tr.transform(rna.loc[eligible_cells].to_numpy())
                Xc = cnv_tr.transform(cnv.loc[eligible_cells].to_numpy())
                Xm = mut_tr.transform(mut.loc[eligible_cells].to_numpy())

                base_mats = {
                    "rna": Xr,
                    "cnv": Xc,
                    "mut": Xm,
                }

                prot_tr_raw = prot_elig.to_numpy()[idx_train]
                prot_al_raw = prot_elig.to_numpy()

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    uses_prot = "prot" in cfg_keys

                    X_nonprot = concat_selected_modalities(
                        mats=base_mats,
                        selected_keys=cfg_keys,
                        n_rows=len(eligible_cells),
                    )

                    # Exact ranked config without proteomics
                    if not uses_prot:
                        if X_nonprot.shape[1] == 0:
                            continue

                        mdl = make_model(cfg_model, SEED)
                        mdl.fit(X_nonprot[idx_train], y_train)
                        pred = mdl.predict(X_nonprot[idx_test])

                        all_bakeoff_rows.append({
                            "seed": run_seed,
                            "config_rank": cfg_rank,
                            "arm": arm,
                            "model": cfg_model,
                            "feature_set": cfg_feature_set,
                            "uses_prot": False,
                            "compound_id": drug,
                            "fold": fold_i,
                            "imputer_strategy": "no_prot_reference",
                            "add_indicators": False,
                            "force_indicators": False,
                            "n_train": n_train,
                            "n_test": n_test,
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })
                        continue

                    # Proteomics-containing ranked config
                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    if len(cfg_keys_wo_prot) > 0:
                        X_ref = concat_selected_modalities(
                            mats=base_mats,
                            selected_keys=cfg_keys_wo_prot,
                            n_rows=len(eligible_cells),
                        )
                        if X_ref.shape[1] > 0:
                            mdl_ref = make_model(cfg_model, SEED)
                            mdl_ref.fit(X_ref[idx_train], y_train)
                            pred_ref = mdl_ref.predict(X_ref[idx_test])

                            all_bakeoff_rows.append({
                                "seed": run_seed,
                                "config_rank": cfg_rank,
                                "arm": arm,
                                "model": cfg_model,
                                "feature_set": cfg_feature_set,
                                "uses_prot": True,
                                "compound_id": drug,
                                "fold": fold_i,
                                "imputer_strategy": "reference_without_prot",
                                "add_indicators": False,
                                "force_indicators": False,
                                "n_train": n_train,
                                "n_test": n_test,
                                "spearman": spearman_corr(y_test, pred_ref),
                                "r2": float(r2_score(y_test, pred_ref)),
                            })

                    force_indicators = (arm == "prot_combined_union")

                    for strat_name, imputer, add_ind in IMPUTER_STRATEGIES:
                        imp_copy = copy.deepcopy(imputer)

                        try:
                            Xp = fit_transform_imputer(
                                X_train=prot_tr_raw,
                                X_all=prot_al_raw,
                                imputer=imp_copy,
                                add_indicators=add_ind,
                                force_indicators=force_indicators,
                                random_state=SEED + 3,
                                n_pca_components=PCA_COMPONENTS,
                            )
                        except Exception as e:
                            print(f"    [WARN] {arm}/{cfg_feature_set}/{strat_name}/fold{fold_i}: {e}")
                            continue

                        if Xp.shape[1] == 0:
                            continue

                        parts = []
                        bad_cfg = False
                        for k in cfg_keys:
                            if k == "prot":
                                parts.append(Xp)
                            else:
                                arr = base_mats.get(k)
                                if arr is None or arr.shape[1] == 0:
                                    bad_cfg = True
                                    break
                                parts.append(arr)

                        if bad_cfg or len(parts) == 0:
                            continue

                        Xf = np.concatenate(parts, axis=1)

                        mdl = make_model(cfg_model, SEED)
                        mdl.fit(Xf[idx_train], y_train)
                        pred = mdl.predict(Xf[idx_test])

                        all_bakeoff_rows.append({
                            "seed": run_seed,
                            "config_rank": cfg_rank,
                            "arm": arm,
                            "model": cfg_model,
                            "feature_set": cfg_feature_set,
                            "uses_prot": True,
                            "compound_id": drug,
                            "fold": fold_i,
                            "imputer_strategy": strat_name,
                            "add_indicators": add_ind,
                            "force_indicators": force_indicators,
                            "n_train": n_train,
                            "n_test": n_test,
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })

        print(f"    drugs_run={n_run}, drugs_skipped={n_skip}")

        arm_rows = [
            r for r in all_bakeoff_rows
            if r["seed"] == run_seed and r["arm"] == arm
        ]
        arm_df = pd.DataFrame(arm_rows)
        arm_ckpt_path = OUT_REPORTS / f"impute_bakeoff_seed{run_seed}_{arm}.csv"
        arm_df.to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}  shape={arm_df.shape}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"impute_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")

bakeoff_df = pd.DataFrame(all_bakeoff_rows)

nan_n = bakeoff_df["spearman"].isna().sum()
if nan_n > 0:
    print(f"\n[INFO] {nan_n} NaN Spearman values, nanmean used in all summaries.")

merged_path = OUT_REPORTS / f"impute_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print(f"\nMerged bake-off: {merged_path}  shape={bakeoff_df.shape}")

drug_means = (
    bakeoff_df
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy", "compound_id"],
        as_index=False,
    )
    .agg(
        spearman=("spearman", lambda x: float(np.nanmean(x))),
        r2=("r2", lambda x: float(np.nanmean(x))),
        n_folds=("fold", "nunique"),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(
        ["config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", lambda x: float(np.nanmean(x))),
        median_spearman=("spearman", lambda x: float(np.nanmedian(x))),
        std_spearman=("spearman", lambda x: float(np.nanstd(x))),
        mean_r2=("r2", lambda x: float(np.nanmean(x))),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

base_ref = (
    bakeoff_summary[
        bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ][
        ["config_rank", "arm", "model", "feature_set", "mean_spearman", "imputer_strategy"]
    ]
    .rename(
        columns={
            "mean_spearman": "baseline_mean_spearman",
            "imputer_strategy": "baseline_strategy",
        }
    )
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

bakeoff_summary = bakeoff_summary.merge(
    base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)

bakeoff_summary["delta_vs_baseline"] = np.where(
    bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"]),
    0.0,
    bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"],
)

summary_path = OUT_REPORTS / f"impute_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
print("\nBake-off summary (3 seeds merged):")
display(bakeoff_summary)

per_seed_summary = (
    drug_means
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", lambda x: float(np.nanmean(x))),
        median_spearman=("spearman", lambda x: float(np.nanmedian(x))),
        std_spearman=("spearman", lambda x: float(np.nanstd(x))),
    )
    .sort_values(["config_rank", "seed", "mean_spearman"], ascending=[True, True, False])
)

per_seed_path = OUT_REPORTS / "impute_bakeoff_per_seed_summary.csv"
per_seed_summary.to_csv(per_seed_path, index=False)
print("\nPer-seed summary:")
display(per_seed_summary)

print("IMPUTER LOCK DECISION")

imputer_choice = {}

for cfg in TOP10_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    cfg_uses_prot = "prot" in parse_feature_set(cfg_feature_set)

    key = f"rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = bakeoff_summary[
        (bakeoff_summary["config_rank"] == cfg_rank) &
        (bakeoff_summary["arm"] == cfg_arm) &
        (bakeoff_summary["model"] == cfg_model) &
        (bakeoff_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        imputer_choice[key] = {
            "chosen_strategy": None,
            "reason": "No bake-off rows produced for this config.",
        }
        continue

    if not cfg_uses_prot:
        ref_rows = cfg_df[cfg_df["imputer_strategy"] == "no_prot_reference"]
        ref_sp = float(ref_rows["mean_spearman"].iloc[0]) if len(ref_rows) else np.nan

        imputer_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": "not_applicable",
            "add_indicators": False,
            "force_indicators_track2": False,
            "mean_spearman_chosen": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "mean_spearman_baseline": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "delta_vs_baseline": 0.0,
            "n_seeds_in_estimate": len(ALL_SEEDS),
            "reason": "This ranked configuration does not contain proteomics, so imputation is not applicable.",
        }
        print(f"\n{key}:")
        print("  Chosen   : not_applicable")
        continue

    candidates = cfg_df[
        ~cfg_df["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ].copy()

    if cfg_arm == "prot_combined_union":
        ind_candidates = candidates[candidates["imputer_strategy"].str.contains("indicator", regex=False)]
        if ind_candidates.shape[0] > 0:
            candidates = ind_candidates

    if candidates.shape[0] == 0:
        imputer_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": None,
            "reason": "No imputation candidates available for this config.",
        }
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]

    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_prot"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan

    chosen = str(best["imputer_strategy"])
    mean_sp = float(best["mean_spearman"])
    std_sp = float(best["std_spearman"])
    delta = float(best["delta_vs_baseline"]) if pd.notna(best["delta_vs_baseline"]) else np.nan

    imputer_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": chosen,
        "add_indicators": ("indicator" in chosen) or (cfg_arm == "prot_combined_union"),
        "force_indicators_track2": (cfg_arm == "prot_combined_union"),
        "mean_spearman_chosen": round(mean_sp, 6),
        "std_spearman_chosen": round(std_sp, 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(delta, 6) if np.isfinite(delta) else None,
        "n_seeds_in_estimate": len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across {len(ALL_SEEDS)} seeds"
            + (
                f"; delta vs config-specific no-prot reference: {delta:+.4f}."
                if np.isfinite(delta) else
                "; no config-specific no-prot reference available."
            )
            + (" Indicators mandatory for combined union arm." if cfg_arm == "prot_combined_union" else "")
        ),
    }

    print(f"\n{key}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}")
    if np.isfinite(base_sp):
        print(f"  Baseline : {base_sp:.4f}  Δ={delta:+.4f}")

imputer_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "pca_components": PCA_COMPONENTS,
    "knn_neighbors": KNN_NEIGHBORS,
    "gpu_accel_requested": USE_GPU_ACCEL,
    "gpu_accel_active": GPU_ACCEL_ACTIVE,
    "top10_configs": TOP10_CONFIGS,
    "note": (
        "Imputer, scaler, PCA, and model are all fitted on training fold only. "
        "No global statistics are used at any point. "
        "For proteomics-containing configs, imputation strategies are compared within the exact ranked config. "
        "For no-prot configs, imputation is marked not_applicable."
    ),
    "config_choices": imputer_choice,
}

imputer_choice_path = OUT_META / "imputer_choice.json"
write_json(imputer_choice_doc, imputer_choice_path)
print(f"\nImputer choice written : {imputer_choice_path}")

global_copy = IN_META / "imputer_choice.json"
write_json(imputer_choice_doc, global_copy)
print(f"Global copy written    : {global_copy}")

IMPUTATION BAKE-OFF (3 seeds)
[resume] Seed 19537 file exists but is from old schema, rerunning.
[resume] Seed 1584678 file exists but is from old schema, rerunning.
[resume] Seed 17052356 file exists but is from old schema, rerunning.
[resume] Seeds remaining: [19537, 1584678, 17052356]
  Seed 19537
  [resume] seed=19537 arm=prot_combined_union checkpoint is old schema, rerunning arm.
  prot_combined_union: GroupKFold(n_splits=10), 679 cells


In [ ]:
print("IMPUTER LOCK DECISION")


imputer_choice = {}
for arm in ACTIVE_ARMS:
    arm_df = bakeoff_summary[
        (bakeoff_summary["arm"] == arm) &
        (bakeoff_summary["imputer_strategy"] != "baseline_no_prot")
    ].copy()

    if arm_df.shape[0] == 0:
        print(f"[WARN] No bake-off rows for {arm}, defaulting to median.")
        imputer_choice[arm] = {
            "chosen_strategy": "median",
            "add_indicators": arm == TRACK2_ARM,
            "reason": "No bake-off data; defaulting to median.",
        }
        continue

    if arm == TRACK2_ARM:
        candidates = arm_df[arm_df["imputer_strategy"].str.contains("indicator")]
        if candidates.shape[0] > 0:
            arm_df = candidates

    best      = arm_df.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = bakeoff_summary[
        (bakeoff_summary["arm"] == arm) &
        (bakeoff_summary["imputer_strategy"] == "baseline_no_prot")
    ]
    base_sp  = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan
    chosen   = best["imputer_strategy"]
    mean_sp  = float(best["mean_spearman"])
    std_sp   = float(best["std_spearman"])
    delta    = float(best["delta_vs_baseline"])

    imputer_choice[arm] = {
        "chosen_strategy":          chosen,
        "add_indicators":           "indicator" in chosen or arm == TRACK2_ARM,
        "force_indicators_track2":  arm == TRACK2_ARM,
        "mean_spearman_chosen":     round(mean_sp, 6),
        "std_spearman_chosen":      round(std_sp, 6),
        "mean_spearman_baseline":   round(base_sp, 6),
        "delta_vs_baseline":        round(delta, 6),
        "n_seeds_in_estimate":      len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across "
            f"{len(ALL_SEEDS)} seeds. Delta vs no-prot baseline: {delta:+.4f}."
            + (" Indicators mandatory for combined union arm."
               if arm == TRACK2_ARM else "")
        ),
    }
    print(f"\n{arm}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}  (baseline={base_sp:.4f}, Δ={delta:+.4f})")

imputer_choice_doc = {
    "locked_at":       datetime.now(timezone.utc).isoformat(),
    "seeds_used":      ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target":  PRIMARY_TARGET,
    "pca_components":  PCA_COMPONENTS,
    "knn_neighbors":   KNN_NEIGHBORS,
    "note": (
        "Imputer, scaler, and PCA all fitted on training fold only. "
        "No global statistics used at any point. "
        "Indicators appended before PCA when add_indicators=True. "
        "Estimates averaged across 3 seeds for robustness."
    ),
    "arm_choices": imputer_choice,
    "global_preprocessing_rules": {
        "scaling":   "StandardScaler fit on train fold only",
        "pca":       f"PCA({PCA_COMPONENTS} max components) fit on train fold only",
        "col_drop":  "Columns all-missing in train fold dropped before imputation",
        "combined_union_indicators": "ALWAYS - structural platform-driven missingness",
        f"{PRIMARY_ARM}_indicators":   imputer_choice.get(PRIMARY_ARM,   {}).get("add_indicators", False),
        f"{SECONDARY_ARM}_indicators": imputer_choice.get(SECONDARY_ARM, {}).get("add_indicators", False),
    },
}

imputer_choice_path = OUT_META / "imputer_choice.json"
write_json(imputer_choice_doc, imputer_choice_path)
print(f"\nImputer choice written : {imputer_choice_path}")

global_copy = IN_META / "imputer_choice.json"
write_json(imputer_choice_doc, global_copy)
print(f"Global copy written    : {global_copy}")


IMPUTER LOCK DECISION

prot_procan_depmapSanger:
  Chosen   : knn
  Spearman : 0.1100 ± 0.0986  (baseline=0.0593, Δ=+0.0507)

prot_ms_ccle_gygi:
  Chosen   : mean
  Spearman : 0.1377 ± 0.1186  (baseline=0.0790, Δ=+0.0586)

prot_combined_union:
  Chosen   : knn+indicators
  Spearman : 0.0718 ± 0.0701  (baseline=0.0397, Δ=+0.0321)

Imputer choice written : artifacts\metadata\notebook 4\imputer_choice.json
Global copy written    : artifacts\metadata\imputer_choice.json
